# Whisper：自动语音识别

这个 Notebook 展示 `Whisper（OpenAI）` 的完整语音识别流程。

内容包括：
- 音频预处理与 Mel 频谱图提取
- Whisper Encoder-Decoder 架构解读
- 与 T5（文本 seq2seq）的类比
- 多语言识别与语言检测演示
- 时间戳对齐与字幕生成
- 语音翻译（语音 → 英文文本）
- WER（词错误率）评估

## 1. 环境准备

```bash
pip install torch transformers datasets librosa soundfile matplotlib numpy
# 或者使用 openai-whisper 官方库
pip install openai-whisper
```

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from transformers import WhisperForConditionalGeneration, WhisperProcessor

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    # whisper-tiny 最轻量，适合本地快速演示
    model_name: str = 'openai/whisper-small'
    # 采样率固定为 16kHz（Whisper 训练标准）
    sampling_rate: int = 16000
    # 每次处理的最大音频长度（秒）
    chunk_length_s: int = 30

cfg = Config()
cfg

## 2. Whisper 架构解读

### 2.1 整体架构

Whisper 是标准的 **Encoder-Decoder Transformer**，与 T5 结构完全类似，只是输入从文本换成了音频特征。

```
音频波形
  ↓ Log-Mel Spectrogram（80 维，25ms 帧）
  ↓ 2层 Conv1D（局部特征提取 + 下采样）
  ↓ Sinusoidal 位置编码
  ↓ Transformer Encoder（与 T5 Encoder 同结构）
  ↓ 音频特征序列
  ↓ Transformer Decoder（Cross-Attention 查询音频特征）
  ↓ 文本 Token 序列（自回归生成）
```

### 2.2 与 T5 的类比

| 维度 | T5 | Whisper |
|------|----|---------|
| Encoder 输入 | 文本 token embedding | Log-Mel 频谱图 → Conv → 音频 embedding |
| Decoder | 自回归文本生成 | 自回归文本生成（相同） |
| Cross-Attention | 文本特征 | 音频特征 |
| 任务前缀 | `summarize:` 等 | `<|transcribe|>` / `<|translate|>` 等特殊 token |

### 2.3 多任务特殊 Token

Whisper 通过解码器的起始 token 序列指定任务：

```
<|startoftranscript|> <|zh|> <|transcribe|> <|notimestamps|> ...
                        ↑语言   ↑任务（转录/翻译）  ↑是否输出时间戳
```

### 2.4 规模与训练数据

| 模型 | 参数 | 训练数据 |
|------|------|----------|
| whisper-tiny | 39M | 680K 小时（99 种语言） |
| whisper-small | 244M | 同上 |
| whisper-medium | 769M | 同上 |
| whisper-large-v3 | 1.5B | 同上 + 优化 |

## 3. 音频预处理与 Mel 频谱图

In [ ]:
# 加载 LibriSpeech 测试集样本（英文朗读）
ds = load_dataset('hf-internal-testing/librispeech_asr_demo', 'clean', split='validation', trust_remote_code=True)
sample = ds[0]

audio_array = np.array(sample['audio']['array'], dtype=np.float32)
sr = sample['audio']['sampling_rate']

print(f'音频采样率：{sr} Hz')
print(f'音频时长  ：{len(audio_array)/sr:.2f} 秒')
print(f'参考文本  ：{sample["text"]}')

In [ ]:
# 可视化波形与 Mel 频谱图
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# 时域波形
t = np.arange(len(audio_array)) / sr
axes[0].plot(t, audio_array, linewidth=0.5, color='steelblue')
axes[0].set_xlabel('时间 (s)')
axes[0].set_ylabel('振幅')
axes[0].set_title('音频波形')

# Mel 频谱图（使用 Processor 提取，保证与 Whisper 训练一致）
processor = WhisperProcessor.from_pretrained(cfg.model_name)
inputs = processor(audio_array, sampling_rate=sr, return_tensors='pt')
mel = inputs['input_features'][0].numpy()  # 80, T

im = axes[1].imshow(mel, aspect='auto', origin='lower', cmap='viridis')
axes[1].set_xlabel('时间帧')
axes[1].set_ylabel('Mel 滤波器组')
axes[1].set_title('Log-Mel 频谱图（80 维）')
plt.colorbar(im, ax=axes[1], label='Log 能量')

plt.tight_layout()
plt.show()
print(f'Mel 特征形状：{mel.shape}  (n_mels=80, T={mel.shape[1]})')

## 4. 模型加载

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(cfg.model_name).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Whisper-small 参数量：{total_params:,}（约 {total_params/1e6:.0f}M）')

## 5. Token 形状分析

In [ ]:
@torch.no_grad()
def inspect_shapes(model, inputs, device):
    input_features = inputs['input_features'].to(device)
    print(f'input_features          -> {tuple(input_features.shape)}  (batch, n_mels, T)')

    encoder_out = model.model.encoder(input_features)
    enc_hidden = encoder_out.last_hidden_state
    print(f'encoder hidden states   -> {tuple(enc_hidden.shape)}  (batch, T/2, hidden_dim)')

    # 解码器第一步：给入 <|startoftranscript|>
    decoder_ids = torch.tensor([[model.config.decoder_start_token_id]], device=device)
    dec_out = model.model.decoder(input_ids=decoder_ids, encoder_hidden_states=enc_hidden)
    print(f'decoder hidden states   -> {tuple(dec_out.last_hidden_state.shape)}  (batch, 1, hidden_dim)')
    print(f'lm_head output（logits）-> (batch, 1, vocab_size={model.config.vocab_size})')


inspect_shapes(model, inputs, device)

## 6. 基本语音识别（转录）

In [ ]:
@torch.no_grad()
def transcribe(model, processor, audio, sr, device, language='english', task='transcribe'):
    inputs = processor(audio, sampling_rate=sr, return_tensors='pt').to(device)

    # forced_decoder_ids 指定任务类型（转录 vs 翻译）和语言
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=language, task=task)

    generated = model.generate(
        inputs['input_features'],
        forced_decoder_ids=forced_decoder_ids,
        max_new_tokens=200,
    )
    return processor.batch_decode(generated, skip_special_tokens=True)[0]


pred_text = transcribe(model, processor, audio_array, sr, device)
ref_text  = sample['text']

print(f'参考文本：{ref_text}')
print(f'识别结果：{pred_text}')

## 7. 带时间戳的识别（字幕生成）

In [ ]:
@torch.no_grad()
def transcribe_with_timestamps(model, processor, audio, sr, device):
    inputs = processor(audio, sampling_rate=sr, return_tensors='pt').to(device)

    # return_timestamps=True 让 Decoder 输出时间戳 token
    generated = model.generate(
        inputs['input_features'],
        return_timestamps=True,
        max_new_tokens=400,
    )
    result = processor.batch_decode(generated, skip_special_tokens=False, decode_with_timestamps=True)
    return result[0]


timestamped = transcribe_with_timestamps(model, processor, audio_array, sr, device)
print('带时间戳的识别结果：')
print(timestamped)

## 8. WER（词错误率）评估

In [ ]:
def compute_wer(reference, hypothesis):
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()

    # 编辑距离（动态规划）
    m, n = len(ref_words), len(hyp_words)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])

    edit_dist = dp[m][n]
    wer = edit_dist / max(len(ref_words), 1)
    return wer, edit_dist, len(ref_words)


wer, edits, n_words = compute_wer(ref_text, pred_text)
print(f'参考词数：{n_words}')
print(f'编辑距离：{edits}')
print(f'WER      ：{wer:.3f}  （越低越好，0 = 完全正确）')

## 9. 批量多样本评估

In [ ]:
@torch.no_grad()
def batch_evaluate(model, processor, dataset, device, n_samples=10):
    wers = []
    results = []

    for i in range(min(n_samples, len(dataset))):
        sample = dataset[i]
        audio  = np.array(sample['audio']['array'], dtype=np.float32)
        sr     = sample['audio']['sampling_rate']
        ref    = sample['text']

        pred = transcribe(model, processor, audio, sr, device)
        wer, _, _ = compute_wer(ref, pred)
        wers.append(wer)
        results.append({'ref': ref, 'pred': pred, 'wer': wer})
        print(f'样本 {i+1:2d}  WER={wer:.3f}  | {ref[:60]}...')

    avg_wer = np.mean(wers)
    print(f'\n平均 WER = {avg_wer:.3f}')

    # WER 分布
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(wers)), wers, color='steelblue', alpha=0.8)
    ax.axhline(avg_wer, color='red', linestyle='--', label=f'平均 WER={avg_wer:.3f}')
    ax.set_title(f'Whisper-small WER 分布（{n_samples} 个样本）')
    ax.set_xlabel('样本编号')
    ax.set_ylabel('WER')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return results


results = batch_evaluate(model, processor, ds, device, n_samples=8)

## 10. Whisper 系列模型对比与定位

### 与其他语音模型的对比

| 模型 | 方法 | 多语言 | 翻译 | 时间戳 |
|------|------|--------|------|--------|
| DeepSpeech | CTC | 否 | 否 | 否 |
| wav2vec 2.0 | 对比自监督 + CTC | 是（multilingual） | 否 | 否 |
| **Whisper** | **Encoder-Decoder seq2seq** | **是（99 种语言）** | **是（→英文）** | **是** |

### Whisper 的局限

- 对中文等形态复杂语言 WER 较英文高
- 实时性不足（large 版本）：需要 streaming 推理
- 对口音、噪声鲁棒性优秀，但极端噪声仍有幻觉（幻听）问题

### 与同目录模型的关联

Whisper 的 Encoder 本质是处理音频的 ViT 变体，Decoder 与 T5/GPT-2 共享自回归生成范式——三种 Transformer 架构在音频领域的融合应用。